# Reproduce all MRL figures

Validate the 41-figure contract and execute the six topic notebooks with a
resource-aware schedule. Lightweight notebooks fan out into isolated child
interpreters. Notebooks with substantial independent condition grids execute in
canonical order and use their own spawn-safe pools. This avoids nested pools while
retaining parallelism where it shortens the critical path.

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
import json, os, subprocess, sys

RUN_PROFILE = os.getenv("MRL_RUN_PROFILE", "reduced")  # reduced | publication | smoke
DEVICE = os.getenv("MRL_DEVICE", "auto")
WORKERS = os.getenv("MRL_WORKERS", "auto")
SAVE_FIGURES = os.getenv("MRL_SAVE_FIGURES", "0") == "1"
OUTPUT_DIR = Path(os.getenv("MRL_OUTPUT_DIR", "generated_figures"))
OVERWRITE = os.getenv("MRL_OVERWRITE", "0") == "1"
RUN_EXTERNAL_DATA = os.getenv("MRL_RUN_EXTERNAL_DATA", "0") == "1"
ALLOW_DATA_DOWNLOADS = os.getenv("MRL_ALLOW_DATA_DOWNLOADS", "0") == "1"
FAN_OUT_CPU = os.getenv("MRL_FAN_OUT", "1") == "1"

HERE = Path.cwd()
ROOT = HERE.parent if HERE.name == "experiments" else HERE
EXP = ROOT / "experiments"
EXECUTED = EXP / ".executed" / RUN_PROFILE
NOTEBOOKS = [
    "00_device_physics_and_trace.ipynb",
    "01_distal_credit_ladder.ipynb",
    "02_sequential_and_scaling.ipynb",
    "03_deep_local_and_faults.ipynb",
    "04_biological_grounding.ipynb",
    "05_extensions.ipynb",
]
OUTER_FAN = NOTEBOOKS[0:2] + NOTEBOOKS[4:5]
INNER_PARALLEL = NOTEBOOKS[2:4] + NOTEBOOKS[5:6]

manifest = json.loads((ROOT / "data" / "publication" / "figure_manifest.json").read_text(encoding="utf-8"))
assert manifest["figure_count"] == len(manifest["figures"]) == 41
assert len({row["filename"] for row in manifest["figures"]}) == 41
assert sum(row["tier"] == "main" for row in manifest["figures"]) == 10

if WORKERS == "auto":
    RESOLVED_WORKERS = max(1, min(6, (os.cpu_count() or 4) - 2))
else:
    RESOLVED_WORKERS = max(1, int(WORKERS))


def execute_notebook(name, *, inner_workers):
    env = os.environ.copy()
    env.update({
        "MRL_RUN_PROFILE": RUN_PROFILE,
        "MRL_DEVICE": DEVICE,
        "MRL_WORKERS": str(inner_workers),
        "MRL_SAVE_FIGURES": "1" if SAVE_FIGURES else "0",
        "MRL_OUTPUT_DIR": str(OUTPUT_DIR),
        "MRL_OVERWRITE": "1" if OVERWRITE else "0",
        "MRL_RUN_EXTERNAL_DATA": "1" if RUN_EXTERNAL_DATA else "0",
        "MRL_ALLOW_DATA_DOWNLOADS": "1" if ALLOW_DATA_DOWNLOADS else "0",
    })
    if inner_workers == 1:
        env["MRL_CHILD_PROCESS"] = "1"
    else:
        env.pop("MRL_CHILD_PROCESS", None)
    EXECUTED.mkdir(parents=True, exist_ok=True)
    command = [
        sys.executable, "-m", "jupyter", "nbconvert", "--to", "notebook", "--execute",
        "--ExecutePreprocessor.kernel_name=mrl-trace-venv",
        "--ExecutePreprocessor.timeout=7200",
        "--output", name, "--output-dir", str(EXECUTED.resolve()),
        str((EXP / name).resolve()),
    ]
    done = subprocess.run(command, cwd=ROOT, env=env, text=True, capture_output=True)
    if done.returncode:
        raise RuntimeError(f"{name} failed:\n{done.stdout}\n{done.stderr}")
    return name


completed = []
if os.getenv("MRL_SKIP_EXECUTION", "0") != "1":
    if FAN_OUT_CPU:
        with ThreadPoolExecutor(max_workers=len(OUTER_FAN)) as pool:
            futures = {pool.submit(execute_notebook, name, inner_workers=1): name
                       for name in OUTER_FAN}
            for future in as_completed(futures):
                future.result()
        completed.extend(OUTER_FAN)
    else:
        completed.extend(execute_notebook(name, inner_workers=1) for name in OUTER_FAN)

    # Each heavy notebook owns the whole worker budget in turn; no nested pools.
    completed.extend(execute_notebook(name, inner_workers=RESOLVED_WORKERS)
                     for name in INNER_PARALLEL)

completed = [name for name in NOTEBOOKS if name in completed]
print({"validated_figures": 41, "active_manuscript_figures": 10,
       "executed": completed, "outer_fan": FAN_OUT_CPU,
       "inner_workers": RESOLVED_WORKERS, "figures_saved": SAVE_FIGURES})

## Interpretation

All offline numerical panels are generated from measured fixtures or live model
calls. Full numerical archives are used only when a topic notebook's explicit
archive opt-in is enabled. Dopamine and EEG panels remain absent until their real
datasets and validated caches are enabled; the driver never replaces them with a
toy or reference raster. Saving remains opt-in.